# Problem 6: Transformer for Speech Emotion Recognition

## Self-Attention Based Emotion Classification (PyTorch)

---

## Problem Statement

**Goal**: Use self-attention to model relationships between ALL time frames simultaneously, capturing global emotional patterns.

### Why Transformers?
- **Parallelizable** - faster training than sequential LSTMs
- **Global context** - every frame attends to every other frame
- **State-of-the-art** in speech processing (Wav2Vec, HuBERT, Whisper)
- **No recurrence** - avoids vanishing gradient problems

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import glob
from pathlib import Path
from tqdm import tqdm
import math
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 2. Data Loading and Feature Extraction

In [ ]:
DATA_DIR = Path("data")

EMOTION_MAP = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised",
}

def parse_ravdess_filename(filepath):
    name = Path(filepath).stem
    parts = name.split("-")
    if len(parts) != 7:
        return None
    return {
        "file_path": filepath,
        "emotion_label": EMOTION_MAP.get(parts[2]),
    }

wav_paths = sorted(glob.glob(str(DATA_DIR / "Actor_*" / "*.wav")))
rows = [r for r in (parse_ravdess_filename(p) for p in wav_paths) if r]
df = pd.DataFrame(rows)

print(f"Found {len(df)} files")

In [ ]:
SAMPLE_RATE = 22050
N_MFCC = 40
MAX_LEN = 200
HOP_LENGTH = 512

def extract_mfcc_sequence(file_path):
    try:
        y, _ = librosa.load(file_path, sr=SAMPLE_RATE)
        mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)
        mfcc = mfcc.T
        
        if mfcc.shape[0] < MAX_LEN:
            mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)), mode='constant')
        else:
            mfcc = mfcc[:MAX_LEN]
        
        return mfcc
    except:
        return None

print("Extracting features...")
X, y = [], []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    features = extract_mfcc_sequence(row['file_path'])
    if features is not None:
        X.append(features)
        y.append(row['emotion_label'])

X = np.array(X)
y = np.array(y)
print(f"Features shape: {X.shape}")

In [ ]:
# Encode and split
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
n_classes = len(label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)

# Normalize
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train.reshape(-1, N_MFCC)).reshape(X_train.shape)
X_val_norm = scaler.transform(X_val.reshape(-1, N_MFCC)).reshape(X_val.shape)
X_test_norm = scaler.transform(X_test.reshape(-1, N_MFCC)).reshape(X_test.shape)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

## 3. PyTorch Dataset

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32
train_loader = DataLoader(EmotionDataset(X_train_norm, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionDataset(X_val_norm, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(EmotionDataset(X_test_norm, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 4. Transformer Components

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
class TransformerEmotionClassifier(nn.Module):
    """
    Transformer encoder for emotion classification.
    """
    
    def __init__(self, input_dim, d_model, nhead, num_layers, num_classes, 
                 dim_feedforward=256, dropout=0.2, max_len=200):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        
        # Global average pooling
        x = x.mean(dim=1)
        
        # Classification
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x


# Create model
model = TransformerEmotionClassifier(
    input_dim=N_MFCC,
    d_model=64,
    nhead=4,
    num_layers=3,
    num_classes=n_classes,
    dim_feedforward=128,
    dropout=0.2,
    max_len=MAX_LEN
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()
    
    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
    
    return total_loss / len(loader), correct / total

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

NUM_EPOCHS = 150
PATIENCE = 20
best_val_loss = float('inf')
patience_counter = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_transformer_model.pt')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(torch.load('best_transformer_model.pt'))
print("\nTraining complete!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('transformer_training_curves.png', dpi=150)
plt.show()

## 6. Evaluation

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Get predictions
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Transformer Emotion Classifier')
plt.tight_layout()
plt.savefig('transformer_confusion_matrix.png', dpi=150)
plt.show()

## 7. Pre-trained Models (Advanced)

For state-of-the-art results, consider using pre-trained speech models:

```python
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

# Wav2Vec 2.0
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=8
)

# HuBERT
from transformers import HubertForSequenceClassification
model = HubertForSequenceClassification.from_pretrained(
    "facebook/hubert-base-ls960",
    num_labels=8
)
```

## 8. Key Takeaways

### Strengths of Transformers for SER
- **Global context** through self-attention
- **Parallelizable** training (faster than RNNs)
- **Pre-trained models** (Wav2Vec, HuBERT) achieve SOTA

### Limitations
- **Data hungry** - need more data than LSTMs
- **Quadratic complexity** - O(n^2) in sequence length
- **Overfitting risk** on small datasets

### Best Practices
1. Start with **smaller Transformers** (2-3 layers, 64 d_model)
2. Use **strong regularization** (dropout, layer norm)
3. Consider **pre-trained models** for best results